In [1]:
from tensorflow.keras.layers import Concatenate
from tensorflow.keras.layers import Input
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow_hub as hub
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import imquality.brisque as brisque
from tensorflow.keras.layers import MaxPooling2D, Flatten, Conv2D, Dense, BatchNormalization, GlobalAveragePooling2D, Dropout, AveragePooling2D, Resizing, Reshape, Multiply
from tensorflow.keras.applications.densenet import DenseNet169
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.resnet50 import ResNet50
from sklearn.metrics import accuracy_score
from tensorflow.keras.applications.regnet import RegNetY320
from tensorflow.keras.applications.efficientnet import EfficientNetB7
from tensorflow.keras.applications.vgg16 import VGG16
import pandas as pd
import pickle
import os
import cv2

In [2]:
trains = []
tests = []
vals = []
cls_trains = []
cls_vals = []
cls_tests = []
for i in range(29):
    train = os.listdir('Hijja Edited/Hijja Edited/' + str(i))
    cls_train = []
    for j in range(len(train)):
        cls_train.append(str(i))
    val = train[-int((0.1 * len(train))): ]
    cls_val = cls_train[-int((0.1 * len(cls_train))): ]
    train = train[0: -int((0.1 * len(train)))]
    cls_train = cls_train[0: -int((0.1 * len(cls_train)))]
    test = train[-int((0.1 * len(train))): ]
    cls_test = cls_train[-int((0.1 * len(cls_train))): ]
    train = train[1: -int((0.1 * len(train)))]
    cls_train = cls_train[1: -int((0.1 * len(cls_train)))]
    for j in range(len(train)):
        train[j] = 'Hijja Edited/Hijja Edited/' + str(i) + '/' + train[j]
    for j in range(len(val)):
        val[j] = 'Hijja Edited/Hijja Edited/' + str(i) + '/' + val[j]
    for j in range(len(test)):
        test[j] = 'Hijja Edited/Hijja Edited/' + str(i) + '/' + test[j]
    trains.append(train)
    vals.append(val)
    tests.append(test)
    cls_trains.append(cls_train)
    cls_vals.append(cls_val)
    cls_tests.append(cls_test)
trains = [j for sub in trains for j in sub]
vals = [j for sub in vals for j in sub]
tests = [j for sub in tests for j in sub]
cls_trains = [j for sub in cls_trains for j in sub]
cls_vals = [j for sub in cls_vals for j in sub]
cls_tests = [j for sub in cls_tests for j in sub]
trains = np.array(trains)
vals = np.array(vals)
tests = np.array(tests)
cls_trains = np.array(cls_trains)
cls_vals = np.array(cls_vals)
cls_tests = np.array(cls_tests)
trains.shape, vals.shape, tests.shape, cls_trains.shape, cls_vals.shape, cls_tests.shape

((10371,), (1269,), (1140,), (10371,), (1269,), (1140,))

In [3]:
df_train = pd.DataFrame({'filename': trains, 'class': cls_trains})
df_val = pd.DataFrame({'filename': vals, 'class': cls_vals})
df_test = pd.DataFrame({'filename': tests, 'class': cls_tests})

In [4]:
image_gen = ImageDataGenerator(
    rescale = 1. / 255
)
train_set = image_gen.flow_from_dataframe(
    df_train,
    target_size = (32, 32),
    batch_size = 32,
    class_mode = 'categorical',
    shuffle = True
)
val_set = image_gen.flow_from_dataframe(
    df_val,
    target_size = (32, 32),
    batch_size = 32,
    class_mode = 'categorical',
    shuffle = True
)
test_set = image_gen.flow_from_dataframe(
    df_test,
    target_size = (32, 32),
    batch_size = 32,
    class_mode = 'categorical',
    shuffle = True
)

Found 10371 validated image filenames belonging to 29 classes.
Found 1269 validated image filenames belonging to 29 classes.
Found 1140 validated image filenames belonging to 29 classes.


In [5]:
# in_shape = (32, 32, 3)
# in_layer = Input(shape = in_shape)

# densenet = DenseNet169(weights = 'imagenet', input_shape = in_shape, include_top = False)
# mobilenet = MobileNetV2(weights = 'imagenet', input_shape = in_shape, include_top = False)
# for layer in mobilenet.layers:
#     layer.trainable = False
# for layer in densenet.layers:
#     layer.trainable = False
# model_mobilenet = mobilenet(in_layer)
# model_mobilenet = GlobalAveragePooling2D()(model_mobilenet)
# output_mobilenet = Flatten()(model_mobilenet)
# model_densenet = densenet(in_layer)
# model_densenet = GlobalAveragePooling2D()(model_densenet)
# output_densenet = Flatten()(model_densenet)
# concatenated = tf.keras.layers.Concatenate()([output_mobilenet, output_densenet])
# X = BatchNormalization()(concatenated)
# X = Dense(58, activation = 'relu')(X)
# X = Dense(29, activation = 'softmax')(X)

# model = tf.keras.models.Model(inputs = in_layer, outputs = X)

In [6]:
def attention_block(input_feature, ratio = 8):
    channel = input_feature.shape[-1]
    squeeze = GlobalAveragePooling2D()(input_feature)
    excitation = Dense(channel // ratio, activation = 'relu')(squeeze)
    excitation = Dense(channel, activation = 'sigmoid')(excitation)
    excitation = Reshape((1, 1, channel))(excitation)
    scale = Multiply()([input_feature, excitation])
    return scale

in_shape = (32, 32, 3)
in_shape_1 = (32, 32, 3)
in_shape_2 = (32, 32, 3)

in_layer = Input(shape = in_shape)
in_layer_1 = Resizing(32, 32)(in_layer)
in_layer_2 = Resizing(32, 32)(in_layer)

densenet = DenseNet169(weights = 'imagenet', input_shape = in_shape, include_top = False)
mobilenet = MobileNetV2(weights = 'imagenet', input_shape = in_shape_1, include_top = False)
regnet = RegNetY320(weights = 'imagenet', input_shape = in_shape_2, include_top = False)

for layer in regnet.layers[0 : -(len(regnet.layers) // 16)]:
    layer.trainable = False
for layer in densenet.layers[0 : -(len(densenet.layers) // 3)]:
    layer.trainable = False
for layer in mobilenet.layers[0 : -(len(mobilenet.layers) // 3)]:
    layer.trainable = False
    
model_densenet = densenet(in_layer)
output_densenet_attention = attention_block(model_densenet)
output_densenet = GlobalAveragePooling2D()(output_densenet_attention)
output_densenet = Flatten()(output_densenet)

model_regnet = regnet(in_layer_1)
output_regnet_attention = attention_block(model_regnet)
output_regnet = GlobalAveragePooling2D()(output_regnet_attention)
output_regnet = Flatten()(output_regnet)

model_mobilenet = mobilenet(in_layer_2)
output_mobilenet_attention = attention_block(model_mobilenet)
output_mobilenet = GlobalAveragePooling2D()(output_mobilenet_attention)
output_mobilenet = Flatten()(output_mobilenet)

concatenated = Concatenate()([output_densenet, output_regnet, output_mobilenet])
X = BatchNormalization()(concatenated)
X = Dense(8192, activation = 'relu')(X)
X = Dropout(0.5)(X)
X = Dense(8196, activation = 'relu')(X)
X = Dropout(0.5)(X)
X = Dense(29, activation = 'linear')(X)

siamese_model = tf.keras.models.Model(inputs = in_layer, outputs = X)

In [7]:
siamese_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 32, 32, 3)]  0           []                               
                                                                                                  
 resizing (Resizing)            (None, 32, 32, 3)    0           ['input_1[0][0]']                
                                                                                                  
 resizing_1 (Resizing)          (None, 32, 32, 3)    0           ['input_1[0][0]']                
                                                                                                  
 densenet169 (Functional)       (None, 1, 1, 1664)   12642880    ['input_1[0][0]']                
                                                                                              

                                                                                                  
 dense_7 (Dense)                (None, 8196)         67149828    ['dropout[0][0]']                
                                                                                                  
 dropout_1 (Dropout)            (None, 8196)         0           ['dense_7[0][0]']                
                                                                                                  
 dense_8 (Dense)                (None, 29)           237713      ['dropout_1[0][0]']              
                                                                                                  
Total params: 282,895,279
Trainable params: 168,235,953
Non-trainable params: 114,659,326
__________________________________________________________________________________________________


In [8]:
rlr = ReduceLROnPlateau(monitor = "val_accuracy", factor = 0.01, patience = 6, verbose = 0, mode = "max", min_delta = 0.01)

In [9]:
optim = Adam(learning_rate = 0.001)
siamese_model.compile(loss = tf.keras.losses.CategoricalCrossentropy(from_logits = True), optimizer = optim, metrics = ['accuracy'])

In [10]:
history = siamese_model.fit(train_set, steps_per_epoch = 10371 // 32, epochs = 500, batch_size = 32, validation_data = val_set, callbacks = [rlr])

Epoch 1/500
324/324 [==============================] - 179s 426ms/step - loss: 3.2794 - accuracy: 0.3615 - val_loss: 6.7353 - val_accuracy: 0.0678 - lr: 0.0010
Epoch 2/500
324/324 [==============================] - 89s 275ms/step - loss: 1.7473 - accuracy: 0.4898 - val_loss: 2.6044 - val_accuracy: 0.3641 - lr: 0.0010
Epoch 3/500
324/324 [==============================] - 91s 282ms/step - loss: 1.7937 - accuracy: 0.4986 - val_loss: 2.7010 - val_accuracy: 0.3184 - lr: 0.0010
Epoch 4/500
324/324 [==============================] - 97s 300ms/step - loss: 1.7086 - accuracy: 0.5337 - val_loss: 39.2740 - val_accuracy: 0.1292 - lr: 0.0010
Epoch 5/500
324/324 [==============================] - 119s 366ms/step - loss: 1.6083 - accuracy: 0.5498 - val_loss: 14.9526 - val_accuracy: 0.1584 - lr: 0.0010
Epoch 6/500
324/324 [==============================] - 118s 364ms/step - loss: 1.4633 - accuracy: 0.5792 - val_loss: 4.1523 - val_accuracy: 0.2522 - lr: 0.0010
Epoch 7/500
324/324 [====================

324/324 [==============================] - 105s 325ms/step - loss: 0.6550 - accuracy: 0.7924 - val_loss: 0.8744 - val_accuracy: 0.7313 - lr: 1.0000e-11
Epoch 52/500
324/324 [==============================] - 112s 345ms/step - loss: 0.6604 - accuracy: 0.7892 - val_loss: 0.8749 - val_accuracy: 0.7289 - lr: 1.0000e-13
Epoch 53/500
324/324 [==============================] - 103s 318ms/step - loss: 0.6436 - accuracy: 0.7968 - val_loss: 0.8762 - val_accuracy: 0.7250 - lr: 1.0000e-13
Epoch 54/500
324/324 [==============================] - 108s 333ms/step - loss: 0.6657 - accuracy: 0.7900 - val_loss: 0.8742 - val_accuracy: 0.7226 - lr: 1.0000e-13
Epoch 55/500
324/324 [==============================] - 107s 332ms/step - loss: 0.6556 - accuracy: 0.7936 - val_loss: 0.8763 - val_accuracy: 0.7329 - lr: 1.0000e-13
Epoch 56/500
324/324 [==============================] - 108s 333ms/step - loss: 0.6510 - accuracy: 0.7927 - val_loss: 0.8814 - val_accuracy: 0.7250 - lr: 1.0000e-13
Epoch 57/500
324/324 [=

Epoch 101/500
324/324 [==============================] - 105s 324ms/step - loss: 0.6404 - accuracy: 0.8003 - val_loss: 0.8773 - val_accuracy: 0.7242 - lr: 1.0000e-29
Epoch 102/500
324/324 [==============================] - 106s 326ms/step - loss: 0.6516 - accuracy: 0.7941 - val_loss: 0.8769 - val_accuracy: 0.7321 - lr: 1.0000e-29
Epoch 103/500
324/324 [==============================] - 107s 329ms/step - loss: 0.6660 - accuracy: 0.7937 - val_loss: 0.8733 - val_accuracy: 0.7297 - lr: 1.0000e-29
Epoch 104/500
324/324 [==============================] - 106s 326ms/step - loss: 0.6634 - accuracy: 0.7939 - val_loss: 0.8748 - val_accuracy: 0.7321 - lr: 1.0000e-29
Epoch 105/500
324/324 [==============================] - 106s 326ms/step - loss: 0.6483 - accuracy: 0.7951 - val_loss: 0.8756 - val_accuracy: 0.7242 - lr: 1.0000e-29
Epoch 106/500
324/324 [==============================] - 104s 321ms/step - loss: 0.6495 - accuracy: 0.7914 - val_loss: 0.8778 - val_accuracy: 0.7297 - lr: 1.0000e-31
Epoc

324/324 [==============================] - 107s 329ms/step - loss: 0.6595 - accuracy: 0.7918 - val_loss: 0.8770 - val_accuracy: 0.7289 - lr: 1.4013e-45
Epoch 151/500
324/324 [==============================] - 106s 328ms/step - loss: 0.6458 - accuracy: 0.7912 - val_loss: 0.8764 - val_accuracy: 0.7281 - lr: 1.4013e-45
Epoch 152/500
324/324 [==============================] - 105s 324ms/step - loss: 0.6338 - accuracy: 0.8006 - val_loss: 0.8794 - val_accuracy: 0.7266 - lr: 1.4013e-45
Epoch 153/500
324/324 [==============================] - 106s 328ms/step - loss: 0.6561 - accuracy: 0.7946 - val_loss: 0.8802 - val_accuracy: 0.7273 - lr: 1.4013e-45
Epoch 154/500
324/324 [==============================] - 104s 322ms/step - loss: 0.6638 - accuracy: 0.7888 - val_loss: 0.8742 - val_accuracy: 0.7273 - lr: 0.0000e+00
Epoch 155/500
324/324 [==============================] - 104s 321ms/step - loss: 0.6472 - accuracy: 0.7920 - val_loss: 0.8735 - val_accuracy: 0.7281 - lr: 0.0000e+00
Epoch 156/500
324/

324/324 [==============================] - 108s 333ms/step - loss: 0.6432 - accuracy: 0.7981 - val_loss: 0.8774 - val_accuracy: 0.7258 - lr: 0.0000e+00
Epoch 200/500
324/324 [==============================] - 107s 331ms/step - loss: 0.6551 - accuracy: 0.7947 - val_loss: 0.8761 - val_accuracy: 0.7266 - lr: 0.0000e+00
Epoch 201/500
324/324 [==============================] - 107s 331ms/step - loss: 0.6587 - accuracy: 0.7876 - val_loss: 0.8785 - val_accuracy: 0.7250 - lr: 0.0000e+00
Epoch 202/500
324/324 [==============================] - 105s 325ms/step - loss: 0.6753 - accuracy: 0.7886 - val_loss: 0.8739 - val_accuracy: 0.7250 - lr: 0.0000e+00
Epoch 203/500
324/324 [==============================] - 105s 323ms/step - loss: 0.6424 - accuracy: 0.7934 - val_loss: 0.8760 - val_accuracy: 0.7305 - lr: 0.0000e+00
Epoch 204/500
324/324 [==============================] - 106s 328ms/step - loss: 0.6654 - accuracy: 0.7895 - val_loss: 0.8809 - val_accuracy: 0.7266 - lr: 0.0000e+00
Epoch 205/500
324/

KeyboardInterrupt: 